# DoclingDocument & Chunker Inspector

Tool for poking at the structured artifact that flows from docling parse → docling-graph chunker → LLM extraction. Useful for:

- Confirming a PDF parsed the way you expect (pages, tables, picture descriptions, body element types)
- Finding orphan elements (numbers/text fragments that should have been part of a table)
- Inspecting how `HybridChunker` slices a document into chunks before the LLM sees them
- Demonstrating cross-page table fragmentation bugs (the SA-2 missile spec table is the running example)

**Default loaded document:** the SA-2 Guideline PDF (`38bebd4a-9137-4f02-ab64-ec08c94b804c`). To inspect a different document, change `DOC_ID` in §1.

**Required services:** the API container (`eip-mmdpp-api-1`) must be reachable at `http://localhost:8005` for the `/docling-raw` endpoint.

## §1 Configuration

In [14]:
import json
import urllib.request
from collections import Counter
from pathlib import Path

# API endpoint for fetching parsed DoclingDocument JSON.
# Default = Docker-internal DNS (works when this notebook runs inside the
# eip-mmdpp-jupyter container, which is the standard setup). If you are running
# Jupyter directly on the host, change to http://localhost:8005.
API_BASE = "http://api:8000"

# Which document to inspect. Default = SA-2 Guideline PDF used as the running example
# for the cross-page table fragmentation bug.
DOC_ID = "38bebd4a-9137-4f02-ab64-ec08c94b804c"

# Chunker configuration matching docker/docling-graph/app/config_builder.py defaults.
CHUNK_MAX_TOKENS = 512
MERGE_PEERS = True

# Optional: dump the loaded JSON to disk for offline poking.
DUMP_PATH = Path("/tmp/docling_doc_inspect.json")


## §2 Load DoclingDocument from the API

Hits `GET /v1/documents/{DOC_ID}/docling-raw`, which returns the parsed-and-enriched DoclingDocument JSON straight from the artifact store. The endpoint adds an `_enrichments` field that isn't part of the DoclingDocument schema — we strip it before validating.

If the parsed JSON is large (>10MB), the request can take 5-30 seconds.

In [15]:
url = f"{API_BASE}/v1/documents/{DOC_ID}/docling-raw"
print(f"Fetching {url} ...")
with urllib.request.urlopen(url, timeout=60) as r:
    raw = r.read()
print(f"Received {len(raw):,} bytes")

doc_json = json.loads(raw)

# Persist for offline use; comment out if undesired.
DUMP_PATH.write_bytes(raw)
print(f"Saved to {DUMP_PATH}")

# Strip the API-added enrichments field before constructing DoclingDocument
# (the model_validate call would reject unknown top-level keys otherwise).
enrichments = doc_json.pop("_enrichments", None)
print(f"_enrichments present: {enrichments is not None} ({type(enrichments).__name__ if enrichments else 'None'})")

Fetching http://api:8000/v1/documents/38bebd4a-9137-4f02-ab64-ec08c94b804c/docling-raw ...
Received 24,457,742 bytes
Saved to /tmp/docling_doc_inspect.json
_enrichments present: True (dict)


In [16]:
# Reconstruct a typed DoclingDocument so we can pass it to the chunker.
from docling_core.types.doc.document import DoclingDocument

doc = DoclingDocument.model_validate(doc_json)
print(f"DoclingDocument: name={doc.name!r}")
print(f"  origin: {doc.origin.filename if doc.origin else 'no-origin'}")
print(f"  pages: {len(doc.pages) if doc.pages else 0}")
print(f"  texts: {len(doc.texts)}")
print(f"  tables: {len(doc.tables)}")
print(f"  pictures: {len(doc.pictures)}")
print(f"  groups: {len(doc.groups) if doc.groups else 0}")

DoclingDocument: name='tmp8_vmc3kn'
  origin: tmp8_vmc3kn.pdf
  pages: 28
  texts: 308
  tables: 2
  pictures: 34
  groups: 6


## §3 Document structure — top-level stats

Element-type counts and per-page distribution. Useful first sniff for whether the doc parsed sensibly.

In [17]:
def page_of(prov):
    """Extract the (1-indexed) page number from a docling Prov entry, or None."""
    if not prov:
        return None
    p = prov[0]
    return getattr(p, "page_no", None) or getattr(p, "page", None)

labels = Counter()
pages_text = Counter()
pages_tables = Counter()
pages_pictures = Counter()

for t in doc.texts:
    labels[str(t.label)] += 1
    p = page_of(t.prov)
    if p is not None:
        pages_text[p] += 1

for tab in doc.tables:
    p = page_of(tab.prov)
    if p is not None:
        pages_tables[p] += 1

for pic in doc.pictures:
    p = page_of(pic.prov)
    if p is not None:
        pages_pictures[p] += 1

print("Text element labels (label -> count):")
for lbl, cnt in labels.most_common():
    print(f"  {lbl}: {cnt}")

print("\nTables on each page:")
for p, c in sorted(pages_tables.items()):
    print(f"  page {p}: {c}")

print("\nPages with the most text elements (top 10):")
for p, c in pages_text.most_common(10):
    print(f"  page {p}: {c} text elements")

Text element labels (label -> count):
  text: 109
  page_header: 60
  page_footer: 56
  list_item: 34
  caption: 28
  section_header: 20
  code: 1

Tables on each page:
  page 6: 1
  page 7: 1

Pages with the most text elements (top 10):
  page 1: 30 text elements
  page 3: 29 text elements
  page 27: 29 text elements
  page 2: 24 text elements
  page 8: 22 text elements
  page 7: 19 text elements
  page 5: 16 text elements
  page 4: 14 text elements
  page 6: 11 text elements
  page 14: 9 text elements


## §4 Tables — render cell content

Walks each table, prints page number and a markdown render of cells.

In [18]:
def render_table_markdown(table) -> str:
    """Lightweight markdown render: walk table.data.grid (or .table_cells), emit pipe-separated rows.

    Falls back to `str(table)` if the structure is unexpected.
    """
    data = getattr(table, "data", None)
    if data is None:
        return f"<no .data on {type(table).__name__}>"
    grid = getattr(data, "grid", None)
    if grid:
        # grid is list[list[TableCell]]
        out = []
        for row in grid:
            cells = [getattr(c, "text", "") or "" for c in row]
            out.append("| " + " | ".join(cells) + " |")
        return "\n".join(out)
    cells = getattr(data, "table_cells", None) or []
    if cells:
        # Project cells onto a (max_row+1) x (max_col+1) grid.
        max_r = max(c.start_row_offset_idx for c in cells)
        max_c = max(c.start_col_offset_idx for c in cells)
        mat = [[""] * (max_c + 1) for _ in range(max_r + 1)]
        for c in cells:
            txt = getattr(c, "text", "") or ""
            mat[c.start_row_offset_idx][c.start_col_offset_idx] = txt
        return "\n".join("| " + " | ".join(row) + " |" for row in mat)
    return f"<no grid or table_cells on {type(data).__name__}>"

for i, tab in enumerate(doc.tables):
    p = page_of(tab.prov)
    md = render_table_markdown(tab)
    print(f"=== TABLE {i} (page {p}) — {md.count(chr(10))+1} rows ===")
    # Truncate long renders
    if len(md) > 2500:
        print(md[:2500] + "\n... [truncated]")
    else:
        print(md)
    print()

=== TABLE 0 (page 6) — 22 rows ===
| Industry Designation | Industry Designation | SA-75 | S-75 | S-75M |  |  | S-75V | S-75V |  |  | S-75M |
| Military Designation | Military Designation | SA-75 | S-75 | S-75 | S-75M1 | S-75M1 | S-75M | S-75M | S-75M2 | S-75M4 | S-75 |
| NATO Designation | NATO Designation | SA-2A | SA-2C | SA-2D | SA-2D | SA-2D | SA-2C | SA-2C | SA-2D | SA-2D | SA-2E |
| Fan Song Variant | Fan Song Variant | RSNA- 75 | RSN-75 | RSN- 75M | RSN- 75V1 | RSN- 75V1 | RSN- 75V | RSN- 75V | RSNA- 75M | RSN- 75M4 | RSN- 75M |
| Max Range | m | 29000 | 34000 | 43000 | 34000 | 43000 | 43000 | 45000 | 56000 | 76000 |  |
| Min Range | m | 8000 | 8000 | 8000 |  | 7000 | 7000 | 7000 | 6000 | 6000 |  |
| Max Alt | m | 22000 | 27000 | 30000 | 27000 | 30000 | 30000 | 30000 | 30000 | 30000 |  |
| Min Alt | m | 3000 | 3000 | 1000 | 500 | 300 | 1000 | 1000 | 100 | 50 | 5000 |
| Vmax appr tgt | m/s | 417 | 417 | 639 | 556 | 639 | 639 |  | 1000 | 1000 |  |
| Vmax reced tgt | m/s |  |  |  

## §5 Page boundaries — find orphan elements

When docling splits a multi-page table, the continuation rows often parse as **standalone text elements** rather than table cells. They look like loose numbers/labels with no surrounding structure.

Pages with an unusually high count of short numeric/label text elements are a signal that a table got fragmented. For the SA-2 doc, look at page 7 — that's where the lost "2nd Stage Weight" row lives as orphan text fragments (`1028`, `1251`, `1257`, ...).

In [19]:
# Helper: short numeric-looking text elements (suspect orphan table cells).
import re
NUMBER_LIKE = re.compile(r"^[\d,\.\s]+$")

def is_orphan_candidate(text: str) -> bool:
    s = (text or "").strip()
    if not s or len(s) > 30:
        return False
    return bool(NUMBER_LIKE.match(s))

# Per-page: count orphan candidates, flag pages with notable runs of them.
page_orphans = Counter()
page_orphan_samples: dict[int, list[str]] = {}
for t in doc.texts:
    if is_orphan_candidate(t.text or t.orig or ""):
        p = page_of(t.prov)
        if p is None:
            continue
        page_orphans[p] += 1
        page_orphan_samples.setdefault(p, []).append((t.text or t.orig or "").strip())

print("Pages with the most orphan-candidate text elements (top 8):")
for p, cnt in page_orphans.most_common(8):
    samples = page_orphan_samples.get(p, [])[:8]
    print(f"  page {p}: {cnt} orphans — sample: {samples}")

Pages with the most orphan-candidate text elements (top 8):
  page 7: 9 orphans — sample: ['1257', '1380', '1380', '1386', '1028', '1251', '1251', '1257']
  page 3: 1 orphans — sample: ['15']
  page 4: 1 orphans — sample: ['11964']
  page 8: 1 orphans — sample: ['50']
  page 14: 1 orphans — sample: ['00']


In [20]:
# Specific check for SA-2: are the expected-but-missing sustain values (1028, 1251, 1257, 1380, 1386, 1399) present as orphan texts?
EXPECTED_SUSTAIN_VALUES = {"1028", "1251", "1257", "1380", "1386", "1399"}
found = []
for t in doc.texts:
    s = (t.text or t.orig or "").strip()
    if s in EXPECTED_SUSTAIN_VALUES:
        found.append((page_of(t.prov), str(t.label), s))

if found:
    print("Found expected sustain values as standalone text elements:")
    for p, lbl, s in found:
        print(f"  page {p}: label={lbl!r} value={s!r}")
    print()
    print("This is the bug fingerprint: numbers that SHOULD be in a table cell have been parsed as plain text.")
else:
    print("No matches found — either the document doesn't have those values, or they're inside a table (not orphaned).")

Found expected sustain values as standalone text elements:
  page 7: label='page_header' value='1257'
  page 7: label='page_header' value='1380'
  page 7: label='page_header' value='1380'
  page 7: label='page_header' value='1386'
  page 7: label='text' value='1028'
  page 7: label='text' value='1251'
  page 7: label='text' value='1251'
  page 7: label='page_header' value='1257'
  page 7: label='text' value='1399'

This is the bug fingerprint: numbers that SHOULD be in a table cell have been parsed as plain text.


## §6 Run the chunker

Uses Docling's `HybridChunker` with the same configuration the docling-graph service uses (`chunk_max_tokens=512`, `merge_peers=True`). The chunker walks the document body in reading order and emits structure-preserving chunks of bounded token count. These chunks are what eventually get stuffed into the LLM prompt.

Note: `HybridChunker` may produce chunks slightly over `chunk_max_tokens` (e.g., a single long table that doesn't have a natural split point). That's by design.

In [21]:
from docling.chunking import HybridChunker

chunker = HybridChunker(
    chunk_max_tokens=CHUNK_MAX_TOKENS,
    merge_peers=MERGE_PEERS,
)

# chunk_iter() returns DocChunk objects (text + meta + doc_items refs).
# We materialize for inspection.
chunks = list(chunker.chunk(doc))
print(f"Chunker produced {len(chunks)} chunks")

Token indices sequence length is longer than the specified maximum sequence length for this model (2011 > 512). Running this sequence through the model will result in indexing errors


Chunker produced 125 chunks


In [24]:
# Inspect each chunk: token count (approximate via str length / 4), page span, content preview.
def chunk_pages(chunk) -> list[int]:
    """Return the set of page numbers covered by a chunk's referenced doc_items."""
    pages = set()
    items = getattr(chunk.meta, "doc_items", None) or []
    for item in items:
        prov = getattr(item, "prov", None) or []
        for p in prov:
            pn = getattr(p, "page_no", None) or getattr(p, "page", None)
            if pn is not None:
                pages.add(pn)
    return sorted(pages)

print(f"{'idx':>4} {'pages':>14} {'~chars':>8}  preview")
print("-" * 100)
for i, c in enumerate(chunks):
    text = c.text
    pages = chunk_pages(c)
    page_str = (
        f"{pages[0]}"
        if len(pages) <= 1
        else f"{pages[0]}..{pages[-1]} ({len(pages)})"
    )
    preview = text.replace("\n", " \\n ")
    print(f"{i:>4} {page_str:>14} {len(text):>8}  {preview!r}")

 idx          pages   ~chars  preview
----------------------------------------------------------------------------------------------------
   0              1      153  '[FIFB-22](https://www.ausairpower.net/raptor.html) \\n [PACRIM WEPS](https://www.ausairpower.net/region.html) \\n - [Ready to win bigger; \\n faster and smarter with'
   1              1      401  'AI?](http://d.adroll.com/click/?adroll_insertion_id=48760b031b457241b2fc010a98a6d01c&adroll_pixalate_click_url=https%3A//adrta.com/c%3Fclid%3Dar%26paid%3Dar%26avid%3D4ZYN5F45WFCBFID26NI42R%26caid%3DHGVJGN57U5HLNJREUOP7UN%26plid%3DXIWONOR5PFHSJNNQJNHGXV%26siteId%3Dausairpower.net%26kv1%3D728x90%26publisherId%3Dpub-8664514669849908%26kv2%3Dhttps%253a%252f%252fwww.ausairpower.net%252fAPA-S-75-Volkhov.'
   2              1      196  'html%26kv3%3D1debdccc062ab7af3be05d10d9f6513b%26kv4%3D136.53.88. \\n 0%26kv7%3DBA%26kv10%3D%5BISP%5D%26kv11%3D8310425266134763789430582991558063169%26kv18%3D%26kv19%3D%5BDevice_ID%5D%26kv24%3DDeskto

## §7 Cross-chunk analysis — does any chunk see both Weight rows?

The bug fingerprint for SA-2: the LLM never sees the 1st Stage Weight row and the 2nd Stage Weight row in the same chunk (or in chunks where both are recognizable as table cells). If the chunker had merged the page-6 table with its page-7 continuation, an LLM looking at that chunk would extract both booster_mass_kg AND sustain_mass_kg correctly.

Below we look for the marker phrases and identify which chunk each ends up in.

In [10]:
# For SA-2, the booster Weight row in the parsed table contains "1135" "1032" etc.
# The lost sustain Weight row's values are 1028 1251 1257 1380 1386 1399.
BOOSTER_VALUES = ["1135", "1032", "1011", "1007"]
SUSTAIN_VALUES = ["1028", "1251", "1257", "1380", "1386", "1399"]

def chunks_containing(chunks, needles):
    out = []
    for i, c in enumerate(chunks):
        hits = [n for n in needles if n in c.text]
        if hits:
            out.append((i, hits, chunk_pages(c)))
    return out

booster_chunks = chunks_containing(chunks, BOOSTER_VALUES)
sustain_chunks = chunks_containing(chunks, SUSTAIN_VALUES)

print("Chunks containing booster-row values:")
for i, hits, pages in booster_chunks:
    print(f"  chunk {i:>3} (pages {pages}): hits={hits}")

print("\nChunks containing sustain-row values:")
for i, hits, pages in sustain_chunks:
    print(f"  chunk {i:>3} (pages {pages}): hits={hits}")

overlap = {i for i, _, _ in booster_chunks} & {i for i, _, _ in sustain_chunks}
print()
if overlap:
    print(f"OVERLAP: chunk(s) {overlap} contain BOTH booster and sustain row values — chunker united them.")
else:
    print("NO OVERLAP: booster and sustain row values live in different chunks.")
    print("This is the bug fingerprint — the LLM seeing the booster Weight chunk has no")
    print("line of sight to the sustain Weight values, and the chunker emitting the sustain")
    print("row has lost the 'Weight kg' row label that would let the LLM map the values.")

Chunks containing booster-row values:
  chunk  52 (pages [6]): hits=['1135', '1032', '1011', '1007']

Chunks containing sustain-row values:
  chunk  53 (pages [6, 7]): hits=['1028', '1251']
  chunk  58 (pages [7, 8]): hits=['1399']

NO OVERLAP: booster and sustain row values live in different chunks.
This is the bug fingerprint — the LLM seeing the booster Weight chunk has no
line of sight to the sustain Weight values, and the chunker emitting the sustain
row has lost the 'Weight kg' row label that would let the LLM map the values.


In [11]:
# Show the actual content of the chunk(s) holding the orphan sustain values, so the
# loss of structure is visible.
for i, hits, pages in sustain_chunks:
    print(f"=== CHUNK {i} (pages {pages}) ===")
    print(chunks[i].text[:1500])
    if len(chunks[i].text) > 1500:
        print(f"... [truncated, full length {len(chunks[i].text)}]")
    print()

=== CHUNK 53 (pages [6, 7]) ===
 2nd Stage. 2nd Stage, 10 = 2nd Stage. 2nd Stage, 11 = 2nd Stage. Diameter, 1 = mm. Diameter, 2 = 500. Diameter, 3 = 500. Diameter, 4 = 500. Diameter, 5 = 500. Diameter, 6 = 500. Diameter, 7 = 500. Diameter, 8 = 500. Diameter, 9 = 500. Diameter, 10 = 500. Diameter, 11 = . Span, 1 = mm. Span, 2 = . Span, 3 = 1691. Span, 4 = 1691. Span, 5 = 1691. Span, 6 = 1691. Span, 7 = 1691. Span, 8 = 1691. Span, 9 = 1691. Span, 10 = 1691. Span, 11 = 
kg
1028
1251
1251
Source: http://www.rzeszow.mm.pl/~jowitek/S-75.html / Vestnik PVO

=== CHUNK 58 (pages [7, 8]) ===
1399
This image is classified as an engineering_drawing with high confidence; a block_diagram is a plausible alternate as it represents functional spatial relationships, but the inclusion of a metric scale and site geometry confirms it as a technical site plan. The image is a top-down schematic layout of a "Typical SA-2 Guideline Battalion Launch Site" (Russian: Схема боевой позиции зрди С-75), depicting a c

## §8 What you'd do next

Concrete follow-ups depending on what you find:

1. **If the chunker's output looks fine but downstream extraction misses values:** the bug is in `_table_facts.py` / `_alias_map.py` (the parser-side fact extraction layer). Inspect `pipeline_pass_outputs.extract_pass_response_json->'table_overlay'->'facts'` for the same doc and compare expected vs actual schema_field counts.
2. **If a chunk holds both Weight row values but the LLM still can't extract them correctly:** the bug is in the **prompt** or in **schema enforcement** (gemma4:31b at high temperature drops fields). Re-run with `temperature=0.0` or stronger format-mode (`force_json_mode=False` to use schema-grammar mode).
3. **If the Weight rows live in separate chunks AND the row label (`Weight | kg`) only appears in one of them:** the bug is in **docling's table parsing across page boundaries** — the continuation row got demoted to plain text on the next page. Fixing this requires either an upstream docling fix or a custom multi-page table stitcher.

Use this notebook as the starting point for any of those investigations: change `DOC_ID` to a different document and re-run §2-§8.